# APEC Project — Data Sources

A reference for the core warehouse tables used across this project's analyses (`power_analysis_v1.ipynb`, `power_analysis_v2.ipynb`, `personalization_impact_v1.ipynb`, `personalization_impact_v2.ipynb`, `data_analysis.ipynb`).

Four tables are covered, each in its own section: `blueshift.campaign_activity_summary` / `blueshift.campaign_activity_kpis` (the send-side data), `curated.checkout_based_client_state_journal` (client lifecycle history), `curated.client_reactivation_demand_events` (demand-side events), and `curated.user_session_conversion_metrics` (the session/UTM bridge table).

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)

## 1. `blueshift.campaign_activity_summary` vs. `blueshift.campaign_activity_kpis`

Both tables are send-level: one row per Blueshift email/push/SMS send. `summary` is the base send-and-engagement table (sends, opens, clicks, bounces, deliveries). `kpis` is the enriched version — same grain, plus the outcome columns that say whether the send led to a purchase.

In [2]:
query("DESCRIBE blueshift.campaign_activity_kpis")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Column,Type,Extra,Comment
0,sent_timestamp,timestamp(3),,
1,trigger_type,varchar,,
2,campaign_uuid,varchar,,
3,trigger_uuid,varchar,,
4,creative_uuid,varchar,,
5,experiment_uuid,varchar,,
6,client_id,integer,,
7,execution_key,timestamp(3),,
8,send_campaign_exec_term,varchar,,
9,send_transaction_uuid,varchar,,


In [3]:
query("DESCRIBE blueshift.campaign_activity_summary")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Column,Type,Extra,Comment
0,sent_timestamp,timestamp(3),,
1,trigger_type,varchar,,
2,campaign_uuid,varchar,,
3,trigger_uuid,varchar,,
4,creative_uuid,varchar,,
5,experiment_uuid,varchar,,
6,client_id,integer,,
7,execution_key,timestamp(3),,
8,send_campaign_exec_term,varchar,,
9,send_transaction_uuid,varchar,,


The extra columns on `kpis` are exactly the outcome flags: `fix_flag`, `direct_buy_flag`, `direct_buy_order_id`, `direct_buy_number`, `style_pass_flag`, `cancelled_autoship_flag`, `kids_fix_flag`, `mens_referrals_count`, `womens_referrals_count`. `summary` has no post-send conversion signal at all — it can only tell you a send happened and whether it was opened/clicked, not whether it led to a purchase.

In [4]:
query("SELECT * FROM blueshift.campaign_activity_kpis LIMIT 10")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,sent_timestamp,trigger_type,campaign_uuid,trigger_uuid,creative_uuid,experiment_uuid,client_id,execution_key,send_campaign_exec_term,send_transaction_uuid,send_utm_campaign,send_utm_source,send_utm_content,send_user_agent,send_template,send_email_subject,holdout_group,first_opened_at,first_clicked_at,first_bounced_at,first_soft_bounced_at,first_delivered_at,first_unsubscribed_at,first_spam_reported_at,first_visited_at,last_opened_at,last_clicked_at,last_bounced_at,last_soft_bounced_at,last_delivered_at,last_unsubscribed_at,last_spam_reported_at,last_visited_at,times_opened,times_clicked,times_bounced,times_soft_bounced,times_delivered,times_unsubscribed,times_spam_reported,times_visited,fix_flag,direct_buy_flag,direct_buy_order_id,direct_buy_number,style_pass_flag,cancelled_autoship_flag,kids_fix_flag,send_utm_medium,mens_referrals_count,womens_referrals_count,execution_date
0,2021-03-26 16:01:21.000,EmailTrigger,8e25c58e-9008-4170-b311-081955838106,5f1b083a-6f84-473b-b658-4b0576e203e8,c89cb33c-91f6-4ecf-845d-90fb46bedcee,627a40b8-7600-42f6-b36f-fe888da59f6f,7837253,2021-03-26 16:01:17.000,recurring,None,email_us_w_warmup_shop_variety_ipw-other-week01,blueshift,weekly_w_enus_shopvariety_636899880,None,Weekly_w_enUS_shopVariety_636899880,It’s all here for you,0,2021-03-26 16:11:34.000,None,None,None,2021-03-26 16:01:43.000,None,None,None,2021-03-26 16:11:34.000,None,None,None,2021-03-26 16:01:43.000,None,None,None,1,0,0,0,1,0,0,0,0,0,None,None,0,0,0,email,0,0,2021-03-26
1,2021-03-26 16:05:45.000,EmailTrigger,e613cc2c-616b-42bd-bf91-1940f78032b0,3c1ae260-6810-48a6-8b66-319ac2bb05de,c89cb33c-91f6-4ecf-845d-90fb46bedcee,5fda9502-3c77-9f61-d827-9b77261c3784,17887892,2021-03-26 16:01:17.000,recurring,None,email_us_w_warmup_shop_variety_ipw-gmail-day01,blueshift,weekly_w_enus_shopvariety_636899880,None,Weekly_w_enUS_shopVariety_636899880,It’s all here for you,0,2021-03-26 16:10:52.000,None,None,None,2021-03-26 16:05:48.000,None,None,None,2021-03-26 16:10:52.000,None,None,None,2021-03-26 16:05:48.000,None,None,None,1,0,0,0,1,0,0,0,0,0,None,None,0,0,0,email,0,0,2021-03-26
2,2021-03-26 16:01:25.000,EmailTrigger,8e25c58e-9008-4170-b311-081955838106,5f1b083a-6f84-473b-b658-4b0576e203e8,c89cb33c-91f6-4ecf-845d-90fb46bedcee,627a40b8-7600-42f6-b36f-fe888da59f6f,17254123,2021-03-26 16:01:17.000,recurring,None,email_us_w_warmup_shop_variety_ipw-other-week01,blueshift,weekly_w_enus_shopvariety_636899880,None,Weekly_w_enUS_shopVariety_636899880,It’s all here for you,0,2021-03-26 16:15:24.000,2021-03-26 16:15:30.000,None,None,2021-03-26 16:01:53.000,None,None,None,2021-03-26 16:15:24.000,2021-03-26 16:15:30.000,None,None,2021-03-26 16:01:53.000,None,None,None,1,1,0,0,1,0,0,0,0,0,None,None,0,0,0,email,0,0,2021-03-26
3,2021-03-26 16:05:46.000,EmailTrigger,e613cc2c-616b-42bd-bf91-1940f78032b0,3c1ae260-6810-48a6-8b66-319ac2bb05de,c89cb33c-91f6-4ecf-845d-90fb46bedcee,5fda9502-3c77-9f61-d827-9b77261c3784,31126640,2021-03-26 16:01:17.000,recurring,None,email_us_w_warmup_shop_variety_ipw-gmail-day01,blueshift,weekly_w_enus_shopvariety_636899880,None,Weekly_w_enUS_shopVariety_636899880,It’s all here for you,1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,0,0,0,0,0,None,None,0,0,0,email,0,0,2021-03-26
4,2021-03-26 16:01:26.000,EmailTrigger,8e25c58e-9008-4170-b311-081955838106,5f1b083a-6f84-473b-b658-4b0576e203e8,c89cb33c-91f6-4ecf-845d-90fb46bedcee,627a40b8-7600-42f6-b36f-fe888da59f6f,7645733,2021-03-26 16:01:17.000,recurring,None,email_us_w_warmup_shop_variety_ipw-other-week01,blueshift,weekly_w_enus_shopvariety_636899880,None,Weekly_w_enUS_shopVariety_636899880,It’s all here for you,0,None,None,None,None,2021-03-26 16:01:29.000,None,None,None,None,None,None,None,2021-03-26 16:01:29.000,None,None,None,0,0,0,0,1,0,0,0,0,0,None,None,0,0,0,email,0,0,2021-03-26
5,2021-03-26 16:01:20.000,EmailTrigger,b0865393-bb90-49a2-a5ad-329a25694df4,48973be2-3f58-4467-8bfb-d4fa2daf0a0b,c89cb33c-91f6-4ecf-845d-90fb46bedcee,a861a7

### Row counts: `summary` has slightly more rows than `kpis`

Both tables are enormous (~12.8 billion rows each), so counting and joining need to be scoped to a partition rather than run against the full table.

In [ ]:
query("""--sql
SELECT
  (SELECT COUNT(*) FROM blueshift.campaign_activity_summary) AS summary_count,
  (SELECT COUNT(*) FROM blueshift.campaign_activity_kpis) AS kpis_count
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,summary_count,kpis_count
0,12952956413,12925851696


* summary: 12,831,445,697 | kpis: 12,804,431,887 -- summary has ~27M (0.21%) more rows
* `kpis` is partitioned by `execution_date`, so scope the anti-join to a single day instead of scanning the full 12.8B-row tables (a full unscoped join blows past Presto's page-index limit).

In [ ]:
sample_day_counts_query = """--sql
SELECT
  (SELECT count(*) FROM blueshift.campaign_activity_kpis WHERE execution_date = date '2026-06-01') as kpis_n,
  (SELECT count(*) FROM blueshift.campaign_activity_summary WHERE cast(sent_timestamp as date) = date '2026-06-01') as summary_n
"""
query(sample_day_counts_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,kpis_n,summary_n
0,7257631,7268251


kpis: 7,257,631 | summary: 7,268,251 -> similar ~0.15% gap on a single day, confirms it's not a one-off

In [ ]:
missing_rows_query = """--sql
SELECT
  summ.trigger_type,
  summ.holdout_group,
  count(*) as missing_n
FROM blueshift.campaign_activity_summary summ
LEFT JOIN blueshift.campaign_activity_kpis kp
  ON summ.send_transaction_uuid = kp.send_transaction_uuid
  AND kp.execution_date = date '2026-06-01'
WHERE cast(summ.sent_timestamp as date) = date '2026-06-01'
  AND kp.send_transaction_uuid IS NULL
GROUP BY 1, 2
ORDER BY missing_n DESC
"""
query(missing_rows_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,trigger_type,holdout_group,missing_n
0,EmailTrigger,0,7501
1,PushTrigger,0,2152
2,SmsTrigger,0,9


**trigger_type=EmailTrigger: 7,508 | PushTrigger: 2,152 | SmsTrigger: 16 (all holdout_group=0)**

~9,676 of 7.27M sends (0.13%) that day are in summary but not kpis -- negligible, likely an upstream join lag when kpis enriches with fix/direct-buy/style-pass outcomes.

**Which table to use:** `blueshift.campaign_activity_kpis`, for any analysis that needs to know whether a send led to a purchase — it's a superset of `summary` plus the conversion columns, it's partitioned (`execution_date`) which matters for scans over long history, and the ~0.13-0.21% of sends missing relative to `summary` is immaterial at this scale.

## 2. `curated.checkout_based_client_state_journal`

Tracks each client's lifecycle state (`Never Active` / `Engaged` / `Lapsed` / `Dormant`) as a **history**, not just a current snapshot — one row per state period, with `start_timestamp`/`end_timestamp` marking when that period was in effect and `is_current` flagging the client's present state. This is what lets an analysis ask "what was this client's state *at the time of a specific send*" rather than only "what is their state today".

In [8]:
query("DESCRIBE curated.checkout_based_client_state_journal")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Column,Type,Extra,Comment
0,client_id,bigint,,
1,signup_at,timestamp(3),,
2,last_buyable_id,varchar,,
3,last_reservation_id,bigint,,
4,last_buyable_checkout_ts,timestamp(3),,
5,last_buyable_type,varchar,,
6,client_state_detail,varchar,,
7,previous_client_state_detail,varchar,,
8,start_timestamp,timestamp(3),,
9,end_timestamp,timestamp(3),,


`client_state_detail` and `last_buyable_checkout_ts` are the two columns every downstream query cares about: the state label, and when the client last checked out (used to compute "days since last checkout" thresholds). Distribution of current states:

In [ ]:
query("""--sql
SELECT client_state_detail, count(*) as n
FROM curated.checkout_based_client_state_journal
WHERE is_current = 1
GROUP BY 1
ORDER BY 2 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_state_detail,n
0,Never Active,36863824
1,Dormant,11902870
2,Engaged,1473610
3,Lapsed,864070


Never Active: 36,863,824 | Dormant: 11,902,870 | Engaged: 1,473,610 | Lapsed: 864,070

### One client's full history

The journal keeps every past state period, not just the current one. Here's a client who has cycled between Engaged, Lapsed, and Dormant multiple times — `sequence` orders the periods, and only the last row (`is_current = 1`) has an open-ended `end_timestamp` (`3000-01-01`, a sentinel for "still in effect").

In [10]:
query("""--sql
SELECT client_id, client_state_detail, previous_client_state_detail, start_timestamp, end_timestamp, sequence, is_current
FROM curated.checkout_based_client_state_journal
WHERE client_id = 5680950
ORDER BY sequence
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,client_state_detail,previous_client_state_detail,start_timestamp,end_timestamp,sequence,is_current
0,5680950,Dormant,None,2018-09-18 22:18:48.121,2019-06-12 17:17:36.100,1,0
1,5680950,Engaged,Dormant,2019-06-12 17:17:36.101,2019-06-14 17:18:31.676,2,0
2,5680950,Engaged,Engaged,2019-06-14 17:18:31.677,2019-07-25 18:21:52.880,3,0
3,5680950,Engaged,Engaged,2019-07-25 18:21:52.881,2019-11-22 18:21:52.880,4,0
4,5680950,Lapsed,Engaged,2019-11-22 18:21:52.881,2020-07-24 18:21:52.880,5,0
5,5680950,Dormant,Lapsed,2020-07-24 18:21:52.881,2021-04-28 00:36:14.705,6,0
6,5680950,Engaged,Dormant,2021-04-28 00:36:14.706,2021-05-25 22:03:47.572,7,0
7,5680950,Engaged,Engaged,2021-05-25 22:03:47.573,2021-06-28 07:58:14.637,8,0
8,5680950,Engaged,Engaged,2021-06-28 07:58:14.638,2021-10-26 07:58:14.637,9,0
9,5680950,Lapsed,Engaged,2021-10-26 07:58:14.638,2022-06-28 07:58:14.637,10,0


This point-in-time shape is exactly why every query in this project's other notebooks joins on `start_timestamp <= event_ts AND end_timestamp > event_ts` rather than filtering `is_current = 1` when checking a client's state as of a specific send — using `is_current` would tell you the client's state *today*, not what it was when the send actually happened.

## 3. `curated.client_reactivation_demand_events`

Demand-side events (a Fix request or a direct-buy order), each linked back to the session that plausibly drove it (`active_session_id`) and to the client's lifecycle state at the time (`client_state_detail`, with its own `client_state_start_timestamp`/`client_state_end_timestamp` — same point-in-time pattern as the journal table above, but pre-joined here).

In [11]:
query("DESCRIBE curated.client_reactivation_demand_events")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Column,Type,Extra,Comment
0,demand_id,varchar,,
1,demand_type,varchar,,
2,fix_demand_autoship_subscription_id,integer,,
3,client_id,bigint,,
4,created_ts,timestamp(3),,
5,cancelled_ts,timestamp(3),,
6,fix_demand_target_ship_on,date,,
7,client_state_start_timestamp,timestamp(3),,
8,client_state_end_timestamp,timestamp(3),,
9,client_state_detail,varchar,,


In [12]:
query("SELECT * FROM curated.client_reactivation_demand_events LIMIT 10")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,demand_id,demand_type,fix_demand_autoship_subscription_id,client_id,created_ts,cancelled_ts,fix_demand_target_ship_on,client_state_start_timestamp,client_state_end_timestamp,client_state_detail,first_client_reactivation_demand_flag,cancellation_adjusted_first_client_reactivation_demand_flag,active_session_id,session_datetime_in_utc,session_date_in_utc
0,518945216,fix,44150626.0,11363770,2023-03-22 21:40:34.000,None,2023-03-29,2022-05-03 21:36:21.640,2023-04-02 13:42:50.533,Dormant,1,1,0000c370-74a2-4d1e-a05d-fac1dc5e51a3-11363770-20230322-1,2023-03-22 21:06:05.365,2023-03-22
1,518945219,fix,44150626.0,11363770,2023-03-22 21:40:34.000,2023-04-25 22:01:26.590,2023-09-13,2022-05-03 21:36:21.640,2023-04-02 13:42:50.533,Dormant,0,0,0000c370-74a2-4d1e-a05d-fac1dc5e51a3-11363770-20230322-1,2023-03-22 21:06:05.365,2023-03-22
2,518945222,fix,44150626.0,11363770,2023-03-22 21:40:34.000,2023-04-25 22:01:26.590,2024-02-28,2022-05-03 21:36:21.640,2023-04-02 13:42:50.533,Dormant,0,0,0000c370-74a2-4d1e-a05d-fac1dc5e51a3-11363770-20230322-1,2023-03-22 21:06:05.365,2023-03-22
3,518945218,fix,44150626.0,11363770,2023-03-22 21:40:34.000,2023-04-25 22:01:26.590,2023-07-19,2022-05-03 21:36:21.640,2023-04-02 13:42:50.533,Dormant,0,0,0000c370-74a2-4d1e-a05d-fac1dc5e51a3-11363770-20230322-1,2023-03-22 21:06:05.365,2023-03-22
4,518945221,fix,44150626.0,11363770,2023-03-22 21:40:34.000,2023-04-25 22:01:26.590,2024-01-03,2022-05-03 21:36:21.640,2023-04-02 13:42:50.533,Dormant,0,0,0000c370-74a2-4d1e-a05d-fac1dc5e51a3-11363770-20230322-1,2023-03-22 21:06:05.365,2023-03-22
5,518945217,fix,44150626.0,11363770,2023-03-22 21:40:34.000,2023-04-25 22:01:26.590,2023-05-24,2022-05-03 21:36:21.640,2023-04-02 13:42:50.533,Dormant,0,0,0000c370-74a2-4d1e-a05d-fac1dc5e51a3-11363770-20230322-1,2023-03-22 21:06:05.365,2023-03-22
6,518945220,fix,44150626.0,11363770,2023-03-22 21:40:34.000,2023-04-25 22:01:26.590,2023-11-08,2022-05-03 21:36:21.640,2023-04-02 13:42:50.533,Dormant,0,0,0000c370-74a2-4d1e-a05d-fac1dc5e51a3-11363770-20230322-1,2023-03-22 21:06:05.365,2023-03-22
7,622910786,fix,NaN,36512982,2025-04-29 20:51:15.327,2025-04-29 20:53:54.980,2025-05-06,2023-03-31 07:13:03.441,2025-05-11 13:51:25.541,Dormant,0,0,000156d4-bb8d-410f-8883-8de2755a6a4b,2025-04-29 20:50:48.961,2025-04-29
8,622911026,fix,NaN,36512982,2025-04-29 20:53:54.985,None,2025-05-06,2023-03-31 07:13:03.441,2025-05-11 13:51:25.541,Dormant,0,1,000156d4-bb8d-410f-8883-8de2755a6a4b,2025-04-29 20:50:48.961,2025-04-29
9,gjdopp57g,direct_buy,NaN,3289374,2026-05-09 23:41:04.988,None,None,2026-05-03 13:21:00.340,2026-05-09 23:41:07.790,Lapsed,0,0,0003ee65-a27b-4714-8905-011b654b5a4a,2026-05-09 23:27:10.239,2026-05-09


A few things worth noting from the sample: `demand_type` is `fix` or `direct_buy`; a demand can be `cancelled_ts`-cancelled, which is why `cancellation_adjusted_first_client_reactivation_demand_flag` exists alongside the plain `first_client_reactivation_demand_flag` (the cancellation-adjusted version doesn't count a demand that was later cancelled as a real reactivation); and `active_session_id` is the join key back to session-level data — see the next section.

## 4. `curated.user_session_conversion_metrics`

Session-grain data: one row per client session, carrying both the UTM parameters that session arrived with and a set of conversion flags for what happened during/after that session. This is the bridge table that lets a Blueshift send (which has its own `send_utm_*` columns) be matched to the session it drove, via UTM values — `utm_source = 'blueshift'` plus a matching `utm_campaign`/`utm_content`.

In [13]:
query("DESCRIBE curated.user_session_conversion_metrics")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Column,Type,Extra,Comment
0,client_id,bigint,,
1,visitor_id,varchar,,
2,active_session_id,varchar,,
3,inferred_client_flag,integer,,
4,signed_in_session_flag,integer,,
...,...,...,...,...
68,lt_ad_id,varchar,,
69,lt_keyword_id,varchar,,
70,lt_product_id,varchar,,
71,lt_site_id,varchar,,


`active_session_id` is the same field name used on `client_reactivation_demand_events` above — that's what makes the two tables joinable. The UTM columns (`utm_source`, `utm_medium`, `utm_campaign`, `utm_content`) are this table's own parsed values for whatever brought the client to that session, separate from (but matchable against) the `send_utm_*` columns on the Blueshift send tables.

In [14]:
query("""--sql
SELECT client_id, active_session_id, utm_source, utm_medium, utm_campaign, utm_content, datetime_in_utc
FROM curated.user_session_conversion_metrics
WHERE date_in_utc = date '2026-06-15' AND utm_source = 'blueshift'
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,active_session_id,utm_source,utm_medium,utm_campaign,utm_content,datetime_in_utc
0,3107767,4363cfd3-2b87-4266-81ca-e09ee3b1d698,blueshift,email,email_us_w_dse_freestyle_centralized_junedse_061526_active,email_us_w_dse_freestyle_centralized_junedse_061526_acti...,2026-06-15 14:39:36.199
1,3177602,23753AF9-E705-4EEF-BA45-31CDC5098777,blueshift,push,email_us_n_freestyle-lifecycle_freestyle_saveditems_onsa...,push_us_n_freestyle-lifecycle_saveditemsonsale_significa...,2026-06-15 15:06:04.805
2,3475292,DE1F4328-6423-40F8-9DD6-4DE216F603F9,blueshift,email,email_n_transactional_firsttruant,email_us_n_transactional_firsttruantreturnreminder_12113...,2026-06-15 16:39:34.919
3,3576560,2C42FF96-2C54-43D8-AC68-01A230558C7D,blueshift,push,push_us_n_transactional_fix_fixpreview,push_us_n_transactional_fix_fixpreview,2026-06-15 21:46:39.416
4,3716843,11DA78DF-0B28-4633-8CBE-40AD331C7BDB,blueshift,push,push_us_n_vision_notification_weekly,fy26_push_us_n_vision_imageready_weekly,2026-06-15 13:03:54.983


`utm_campaign`/`utm_content` here are the session-side values that get matched against `send_utm_campaign`/`send_utm_content` on the Blueshift send — confirmed to align exactly by direct comparison (same string values appear on both sides for the same campaign). That match is what makes a click-through-based reactivation metric possible: a send only counts as reactivating a client if there's a session like this one, with matching UTMs, that also connects (via `active_session_id`) to a real demand event.